1. 导入所有需要的库

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
import numpy as np

2. 读取文件。
- 若无法找到这个文件，则报错：“This File is not found.”。
- 读取时跳过前19行，且仅保留Target、Sample和Cq这三列的数据。因为前19行数据与后续分析无关，因此需要过滤掉。

In [ ]:
try:
    df = pd.read_csv("admin_2025-12-10 14-24-55_782BR20773.csv", skiprows=19, usecols=[2, 4, 5])
except FileNotFoundError:
    print("This File is not found.")

3. 数据预处理
- 将读取的数据按照Target分为3组。本数据来源于一块96孔板，包括3个不同的基因（1个为内参基因，另外2个为想要知道表达情况的目的基因），按Target分组可以将3个不同基因扩增的Cq值分开。
- 再将每组根据Sample进行排序。根据Sample排序可以使得复孔一一对应，便于后续处理。
- 将排序后的Cq值（数据格式为`Series`）存入新变量中。用于后续处理。

In [ ]:
df1 = df[df['Target'] == 'Rp49'].copy()
df2 = df[df['Target'] == 'CG12896'].copy()
df3 = df[df['Target'] == 'CG31636'].copy()

df1_sorted = df1.sort_values(by='Sample')
df2_sorted = df2.sort_values(by='Sample')
df3_sorted = df3.sort_values(by='Sample')

Ser1 = df1_sorted['Cq']
Ser2 = df2_sorted['Cq']
Ser3 = df3_sorted['Cq'] 

4. 数据处理
- 计算 $\Delta Cq$ ：利用numpy库的向量计算功能，计算出 $\Delta Cq$

In [ ]:
deltaCq1 = (Ser2.values - Ser1.values).tolist()
deltaCq2 = (Ser3.values - Ser1.values).tolist()

- 计算 $2^{-\Delta Cq}$ 并将其存进新的list

In [ ]:
def two_neg_deltaCq(list1, list2):
    for _ in list1:
        _ = float(_)
        _ = 2 ** (-_)
        list2.append(_)

two_neg_deltaCq1 = []
two_neg_deltaCq2 = []
two_neg_deltaCq(deltaCq1, two_neg_deltaCq1)
two_neg_deltaCq(deltaCq2, two_neg_deltaCq2)

- 计算每个Target中，对照组的 $2^{-\Delta Cq}$ 的均值，用于后续的均一化（Normalization）

In [ ]:
mean_2_neg_deltaCq1 = sum(two_neg_deltaCq1[:9])/len(two_neg_deltaCq1[:9])
mean_2_neg_deltaCq2 = sum(two_neg_deltaCq2[:9])/len(two_neg_deltaCq2[:9])

- 计算均一化后的 $2^{-\Delta Cq}$ 值，并存入新的list中

In [ ]:
def Nor_2_neg_deltaCq(list1, list2, mean):
    for _ in list1:
        _ = _ / mean
        list2.append(_)

Nor_2_neg_deltaCq1 = []
Nor_2_neg_deltaCq2 = []
Nor_2_neg_deltaCq(two_neg_deltaCq1, Nor_2_neg_deltaCq1, mean_2_neg_deltaCq1)
Nor_2_neg_deltaCq(two_neg_deltaCq2, Nor_2_neg_deltaCq2, mean_2_neg_deltaCq2)

5. 数据转化与输出
- 将存有均一化后的 $2^{-\Delta Cq}$ 值的list转成DataFrame格式，并保存为新的csv文件。
- 本步骤获得的csv文件数据可直接用于`GraphPad Prism`绘图。

In [ ]:
groups = ['DMSO', 'PFOA 0.01mg/L', 'PFOA 1mg/L']
final_df1 = pd.DataFrame({
    'Group': np.repeat(groups, 9),
    'Value': Nor_2_neg_deltaCq1
})
final_df2 = pd.DataFrame({
    'Group': np.repeat(groups, 9),
    'Value': Nor_2_neg_deltaCq2
})

final_df1.to_csv(f'{df2['Target'].iloc[0]}_qPCR_result.csv')
final_df2.to_csv(f'{df3['Target'].iloc[0]}_qPCR_result.csv')

6. 绘图
- 将DataFrame数据提取成Series格式，按照处理分为3组，在本数据中，DMSO组为对照组，PFOA两个浓度为处理组。

In [ ]:
group_DMSO_1 = final_df1[final_df1['Group'] == 'DMSO']['Value']
group_PFOA_001_1 = final_df1[final_df1['Group'] == 'PFOA 0.01mg/L']['Value']
group_PFOA_1_1 = final_df1[final_df1['Group'] == 'PFOA 1mg/L']['Value']

group_DMSO_2 = final_df2[final_df2['Group'] == 'DMSO']['Value']
group_PFOA_001_2 = final_df2[final_df2['Group'] == 'PFOA 0.01mg/L']['Value']
group_PFOA_1_2 = final_df2[final_df2['Group'] == 'PFOA 1mg/L']['Value']

- 进行t-test检验

In [ ]:
_, p_val_001_1 = stats.ttest_ind(group_DMSO_1, group_PFOA_001_1)
_, p_val_1_1 = stats.ttest_ind(group_DMSO_1, group_PFOA_1_1)

_, p_val_001_2 = stats.ttest_ind(group_DMSO_2, group_PFOA_001_2)
_, p_val_1_2 = stats.ttest_ind(group_DMSO_2, group_PFOA_1_2)

- 将p值转化为*号和ns，后续绘图中用于标注显著性。

In [ ]:
def pvalue2asterisks(pvalue):
    if pvalue <= 0.0001:
        return "****"
    elif pvalue <= 0.001:
        return "***"
    elif pvalue <= 0.01:
        return "**"
    elif pvalue <= 0.05:
        return "*"
    else:
        return "ns"

- 绘制图像。绘制为散点图叠加条形图的图像，并标注显著性和bar。

In [ ]:
plt.figure(figsize=(8, 6))
sns.set_style("ticks")

ax = sns.barplot(x='Group', y='Value', data=final_df1, 
                 palette='coolwarm', 
                 width=0.5,
                 capsize=0.1,   # 添加误差线顶端的横杠
                 errwidth=1.5,  # 误差线宽度
                 ci=68,         # 68% CI ≈ SEM; 设为 'sd' 则为标准差
                 alpha=0.8)     # 条形透明度

sns.stripplot(x='Group', y='Value', data=final_df1,
              color='black',
              alpha=0.6,
              size=6,
              jitter=True,
              ax=ax)

y_max = final_df1['Value'].max()
h = y_max * 0.05

x1, x2 = 0, 1
plt.plot([x1, x2],
         [y_max + 5*h, y_max + 5*h],
         lw=1.5, c='k')

plt.text((x1+x2)*.5, y_max + 6*h, pvalue2asterisks(p_val_001_1),
         ha='center', va='bottom', color='k')

x1, x2 = 0, 2
# 注意高度要比前一个更高，避免重叠
plt.plot([x1, x2], 
         [y_max + 8*h, y_max + 8*h], 
         lw=1.5, c='k')
plt.text((x1+x2)*.5, y_max + 9*h, pvalue2asterisks(p_val_1_1), 
         ha='center', va='bottom', color='k')

plt.xlabel('')
plt.ylabel('Relative amount of mRNA (Normalized to Rp49)')

plt.ylim(0, y_max * 2)

sns.despine()

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.set_style("ticks")

ax = sns.barplot(x='Group', y='Value', data=final_df1, 
                 palette='coolwarm', 
                 width=0.5,
                 capsize=0.1,   # 添加误差线顶端的横杠
                 errwidth=1.5,  # 误差线宽度
                 ci=68,         # 68% CI ≈ SEM; 设为 'sd' 则为标准差
                 alpha=0.8)     # 条形透明度

sns.stripplot(x='Group', y='Value', data=final_df1,
              color='black',
              alpha=0.6,
              size=6,
              jitter=True,
              ax=ax)

y_max = final_df1['Value'].max()
h = y_max * 0.05

x1, x2 = 0, 1
plt.plot([x1, x2],
         [y_max + 5*h, y_max + 5*h],
         lw=1.5, c='k')

plt.text((x1+x2)*.5, y_max + 6*h, pvalue2asterisks(p_val_001_2),
         ha='center', va='bottom', color='k')

x1, x2 = 0, 2
# 注意高度要比前一个更高，避免重叠
plt.plot([x1, x2], 
         [y_max + 8*h, y_max + 8*h], 
         lw=1.5, c='k')
plt.text((x1+x2)*.5, y_max + 9*h, pvalue2asterisks(p_val_1_2), 
         ha='center', va='bottom', color='k')

plt.xlabel('')
plt.ylabel('Relative amount of mRNA (Normalized to Rp49)')

plt.ylim(0, y_max * 2)

sns.despine()

plt.show()